In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:

%run ../Setup/utililties


In [0]:
# Creating the UI widgets for catalog & data source
dbutils.widgets.text("Catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "data_source")

catalog = dbutils.widgets.get("Catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
#csv file in s3 
base_path = f's3://fmcg-data-lake-sportsbar1/{data_source}/*.csv'
print(base_path)

s3://fmcg-data-lake-sportsbar1/customers/*.csv


## Bronze

In [0]:
df= (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("load_date", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
    .withColumn("customer_id", F.col("customer_id").cast("string"))
)


In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- load_date: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(df.limit(5))

customer_id,customer_name,city,load_date,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-08-14T15:32:11.230Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-08-14T15:32:11.230Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-08-14T15:32:11.230Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-08-14T15:32:11.230Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-08-14T15:32:11.230Z,customers.csv,1404


In [0]:
#Writing the data to delta table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite").saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5122603299803824>, line 5
      1 #Writing the data to delta table
      2 df.write\
      3     .format("delta")\
      4     .option("delta.enableChangeDataFeed", "true")\
----> 5     .mode("overwrite").saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(se

## Silver

#### Change customer_id to string

In [0]:
#Changing the type of customer_id to string since it's kind of a categorical data
df = df.withColumn("customer_id", F.col("customer_id").cast("string"))

#### Drop duplicates

In [0]:
# Checking if there are any duplicates

df_dups = df.groupBy("customer_id").count().filter("count > 1")
display(df_dups)


customer_id,count
789522,2
789321,2
789503,2
789603,2


In [0]:
#dropping duplicates

df_silver=df.dropDuplicates(["customer_id"])


#### Trim spaces (customer_name)

In [0]:
display( df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

customer_id,customer_name,city,load_date,file_name,file_size
789521,PrimeFuel Nutrition,null,2026-08-14T15:32:30.539Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-08-14T15:32:30.539Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-08-14T15:32:30.539Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-08-14T15:32:30.539Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-08-14T15:32:30.539Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-08-14T15:32:30.539Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

#### Typos(city)

In [0]:
df_silver.select("city").distinct().show()

+----------+
|      city|
+----------+
| Hyderabad|
|  NewDheli|
|      NULL|
| Bengaluru|
| New Delhi|
| Bengalore|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|Hyderabadd|
+----------+



In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


correct = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(correct), F.col("city"))
         .otherwise(None)
    )
)
df_silver.select("city").distinct().show()

+---------+
|     city|
+---------+
|Hyderabad|
|New Delhi|
|     NULL|
|Bengaluru|
+---------+



#### Title Case (Eg: Hub & hub)

In [0]:
df_silver.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|   SprintX Nutrition|
| PrimeFuel Nutrition|
|      FitFuel Market|
|       Recovery Lane|
|     Endurance Foods|
|MacroBite Superfoods|
|      PowerSnack hub|
|      PowerSnack Hub|
|    ZenAthlete Foods|
|   champion's Choice|
|   SprintX nutrition|
|      StaminaX Store|
|Peak Performance ...|
|      GamePlan Foods|
|Athlete's Choice ...|
|Peak performance ...|
|   Champion's choice|
|MacroBite superfoods|
|EliteAthlete Nutr...|
|HydroBoost Nutrition|
+--------------------+
only showing top 20 rows


In [0]:
df_silver = df_silver.withColumn("customer_name", F.initcap(F.col("customer_name")))
df_silver.select("customer_name").distinct().show()


+--------------------+
|       customer_name|
+--------------------+
|   Sprintx Nutrition|
| Primefuel Nutrition|
|      Fitfuel Market|
|       Recovery Lane|
|     Endurance Foods|
|Macrobite Superfoods|
|      Powersnack Hub|
|    Zenathlete Foods|
|   Champion's Choice|
|      Staminax Store|
|Peak Performance ...|
|      Gameplan Foods|
|Athlete's Choice ...|
|Eliteathlete Nutr...|
|Hydroboost Nutrition|
+--------------------+



#### Missing cities

In [0]:
df_silver.filter(F.col("city").isNull()).show()

+-----------+-------------------+----+--------------------+-------------+---------+
|customer_id|      customer_name|city|           load_date|    file_name|file_size|
+-----------+-------------------+----+--------------------+-------------+---------+
|     789521|Primefuel Nutrition|NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789603|      Recovery Lane|NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789403|  Sprintx Nutrition|NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789420|   Zenathlete Foods|NULL|2026-08-14 15:32:...|customers.csv|     1404|
+-----------+-------------------+----+--------------------+-------------+---------+



In [0]:
df_silver.select(F.col("customer_id")).filter(F.col("city").isNull()).show()

+-----------+
|customer_id|
+-----------+
|     789521|
|     789603|
|     789403|
|     789420|
+-----------+



In [0]:
null_customers = ["Primefuel Nutrition","Recovery Lane","Sprintx Nutrition","Zenathlete Foods"]
df_silver.filter(F.col("customer_name").isin(null_customers)).sort("customer_name").show()

+-----------+-------------------+---------+--------------------+-------------+---------+
|customer_id|      customer_name|     city|           load_date|    file_name|file_size|
+-----------+-------------------+---------+--------------------+-------------+---------+
|     789521|Primefuel Nutrition|     NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789520|Primefuel Nutrition|Bengaluru|2026-08-14 15:32:...|customers.csv|     1404|
|     789522|Primefuel Nutrition|New Delhi|2026-08-14 15:32:...|customers.csv|     1404|
|     789601|      Recovery Lane|Bengaluru|2026-08-14 15:32:...|customers.csv|     1404|
|     789603|      Recovery Lane|     NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789401|  Sprintx Nutrition|Bengaluru|2026-08-14 15:32:...|customers.csv|     1404|
|     789403|  Sprintx Nutrition|     NULL|2026-08-14 15:32:...|customers.csv|     1404|
|     789402|  Sprintx Nutrition|Hyderabad|2026-08-14 15:32:...|customers.csv|     1404|
|     789421|   Zenat

In [0]:
# Let's assume this is the real cities of each missing customer
df_silver = (
    df_silver
    .withColumn(
        "city",
        F.when(F.col("customer_id") == "789521", "Hyderabad")
         .when(F.col("customer_id") == "789603", "New Delhi")
         .when(F.col("customer_id") == "789403", "New Delhi")
         .when(F.col("customer_id") == "789420", "Bengaluru")
         .otherwise(F.col("city"))
    )
)
df_silver.filter(F.col("city").isNull()).show() #NUll values
df_silver.filter(F.col("customer_name").isin(null_customers)).sort("customer_name").show()


+-----------+-------------+----+---------+---------+---------+
|customer_id|customer_name|city|load_date|file_name|file_size|
+-----------+-------------+----+---------+---------+---------+
+-----------+-------------+----+---------+---------+---------+

+-----------+-------------------+---------+--------------------+-------------+---------+
|customer_id|      customer_name|     city|           load_date|    file_name|file_size|
+-----------+-------------------+---------+--------------------+-------------+---------+
|     789521|Primefuel Nutrition|Hyderabad|2026-08-14 15:32:...|customers.csv|     1404|
|     789520|Primefuel Nutrition|Bengaluru|2026-08-14 15:32:...|customers.csv|     1404|
|     789522|Primefuel Nutrition|New Delhi|2026-08-14 15:32:...|customers.csv|     1404|
|     789601|      Recovery Lane|Bengaluru|2026-08-14 15:32:...|customers.csv|     1404|
|     789603|      Recovery Lane|New Delhi|2026-08-14 15:32:...|customers.csv|     1404|
|     789401|  Sprintx Nutrition|Be

#### Standardizing customers table to matche the parent company Data model

In [0]:
df_silver = (
    df_silver
    .withColumn (
        "customer",
        F.concat_ws("-", "customer_name", F.lit("city"))
    )
    # Adding the three other columns
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))

    #Renaming custer_id to customer_code
    .withColumnRenamed("customer_id", "customer_code")
)

In [0]:
df_silver.show(5)

+-------------+-------------------+---------+--------------------+-------------+---------+--------------------+------+----------+-----------+
|customer_code|      customer_name|     city|           load_date|    file_name|file_size|            customer|market|  platform|    channel|
+-------------+-------------------+---------+--------------------+-------------+---------+--------------------+------+----------+-----------+
|       789402|  Sprintx Nutrition|Hyderabad|2026-08-14 15:32:...|customers.csv|     1404|Sprintx Nutrition...| India|Sports Bar|Acquisition|
|       789522|Primefuel Nutrition|New Delhi|2026-08-14 15:32:...|customers.csv|     1404|Primefuel Nutriti...| India|Sports Bar|Acquisition|
|       789202|     Fitfuel Market|Hyderabad|2026-08-14 15:32:...|customers.csv|     1404| Fitfuel Market-city| India|Sports Bar|Acquisition|
|       789521|Primefuel Nutrition|Hyderabad|2026-08-14 15:32:...|customers.csv|     1404|Primefuel Nutriti...| India|Sports Bar|Acquisition|
|     

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold=df_silver.select("customer_code","customer_name", "city","customer","market","platform","channel")


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.ch_dim_{data_source}")

## Merging data source with parent data

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.ch_dim_customers").select(
    "customer_code",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
#upsert operation

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]